# 16 — Network Metrics Deep Dive

**Author:** Bob Mathew D. Sunga, MSc  
**Project:** Philippine Election 2025 Network Science Analysis

This notebook makes the network-science layer explicit. Earlier notebooks already compute degree, weighted degree, PageRank, betweenness centrality, closeness centrality, eigenvector centrality, connected components, density, and community structure. This notebook consolidates those metrics, explains them, compares candidate centrality with actual Senate vote outcomes, and exports report-ready tables and visuals.

The purpose is to prevent the project from looking like a simple election analytics exercise. The center of the work is the structure of relationships: who connects to whom, which hashtags bridge topics, which candidates are central in the discourse network, and which online centrality signals align or diverge from actual votes.


## Visible progress tracker

This notebook uses visible status messages so long runs do not feel silent. It is designed to run after notebooks 06 to 10, because it uses the network metric tables produced there.


In [ ]:
# VISIBLE PROGRESS TRACKER
from pathlib import Path as _ProgressPath
import sys as _ProgressSys
_PROGRESS_ROOT = _ProgressPath.cwd()
if _PROGRESS_ROOT.name == "notebooks":
    _PROGRESS_ROOT = _PROGRESS_ROOT.parent
if str(_PROGRESS_ROOT / "src") not in _ProgressSys.path:
    _ProgressSys.path.insert(0, str(_PROGRESS_ROOT / "src"))
try:
    from election_network_progress import make_tracker
    progress = make_tracker("16_network_metrics_deep_dive", total_steps=11, root=_PROGRESS_ROOT)
except Exception:
    class _FallbackProgress:
        def __init__(self): self.i = 0
        def step(self, label):
            self.i += 1
            print(f"[16_network_metrics_deep_dive] Step {self.i}: {label}")
        def done(self, label="Done"):
            print(f"[16_network_metrics_deep_dive] {label}")
    progress = _FallbackProgress()


## 0. Setup

This notebook loads already generated centrality tables from `data/processed/` and exports additional deep-dive outputs to `outputs/tables/`, `outputs/figures/`, `outputs/interactive/`, and `outputs/report_assets/`.


In [ ]:
progress.step("0. Setup")
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px

try:
    from IPython.display import display, Markdown
except Exception:
    display = print
    def Markdown(x): return x

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from election_network_utils import ensure_dirs, save_plotly, normalize_text

PATHS = ensure_dirs(ROOT)
RAW = PATHS["raw"]
PROCESSED = PATHS["processed"]
TABLES = PATHS["tables"]
FIGURES = PATHS["figures"]
INTERACTIVE = PATHS["interactive"]
REPORT_ASSETS = PATHS["report_assets"]

px.defaults.template = "plotly_white"
print("Project root:", ROOT)
print("Processed data folder:", PROCESSED)


## 1. Load network metric tables

The notebook accepts both the newer repository filenames and earlier generated filenames. This makes the deep-dive notebook usable across the different project builds produced during development.


In [ ]:
progress.step("1. Load network metric tables")

def load_first_available(candidates, folder=PROCESSED, required=False):
    for name in candidates:
        path = folder / name
        if path.exists():
            print(f"Loaded: {path.relative_to(ROOT)}")
            return pd.read_csv(path)
    if required:
        raise FileNotFoundError("Missing required file. Tried: " + ", ".join(candidates))
    print("Missing optional table. Tried:", ", ".join(candidates))
    return pd.DataFrame()

candidate_metrics = load_first_available([
    "candidate_comention_node_metrics.csv",
    "candidate_comention_metrics.csv",
])

hashtag_metrics = load_first_available([
    "hashtag_node_metrics.csv",
    "hashtag_metrics.csv",
])

candidate_hashtag_metrics = load_first_available([
    "candidate_hashtag_bipartite_node_metrics.csv",
    "candidate_hashtag_metrics.csv",
])

candidate_projection_metrics = load_first_available([
    "candidate_projection_shared_hashtags_metrics.csv",
])

comparison = load_first_available([
    "candidate_rank_comparison_online_vs_votes.csv",
    "comparison.csv",
])

national = load_first_available([
    "senate_national.csv",
])

candidate_edges = load_first_available([
    "candidate_comention_edges.csv",
])

hashtag_edges = load_first_available([
    "hashtag_edges_filtered_weight2.csv",
    "hashtag_edges_filtered.csv",
])

print("\nTable shapes:")
for name, df in {
    "candidate_metrics": candidate_metrics,
    "hashtag_metrics": hashtag_metrics,
    "candidate_hashtag_metrics": candidate_hashtag_metrics,
    "candidate_projection_metrics": candidate_projection_metrics,
    "comparison": comparison,
    "national": national,
}.items():
    print(f"{name}: {df.shape}")


## 2. Network metric glossary

The table below clarifies what each metric means in this specific election-discourse project. This is important because centrality is not automatically the same as popularity or voter support.


In [ ]:
progress.step("2. Network metric glossary")
metric_glossary = pd.DataFrame([
    {
        "metric": "degree",
        "plain_meaning": "Number of unique connections a node has.",
        "project_interpretation": "Breadth of association. For candidates, how many other candidates they were co-mentioned with. For hashtags, how many other hashtags they co-appeared with.",
        "important_caution": "High degree means many connections, not necessarily positive support."
    },
    {
        "metric": "weighted_degree",
        "plain_meaning": "Total strength of a node's connections after edge weights are considered.",
        "project_interpretation": "Intensity of repeated association. A candidate repeatedly mentioned with other candidates has high weighted degree.",
        "important_caution": "Strong repeated association may come from support, criticism, controversy, or campaign framing."
    },
    {
        "metric": "degree_centrality",
        "plain_meaning": "Degree normalized by the size of the network.",
        "project_interpretation": "Comparable breadth of connection across networks of different sizes.",
        "important_caution": "Useful for comparing within a network, but less meaningful across very different network types."
    },
    {
        "metric": "pagerank",
        "plain_meaning": "Importance based on being connected to other important nodes.",
        "project_interpretation": "Structural prominence. A candidate or hashtag becomes important if it is linked to other central candidates or hashtags.",
        "important_caution": "PageRank captures discourse prominence, not guaranteed electoral persuasion."
    },
    {
        "metric": "betweenness_centrality",
        "plain_meaning": "How often a node sits on the shortest paths between other nodes.",
        "project_interpretation": "Bridge or broker role. A high-betweenness hashtag or candidate connects otherwise separate conversations.",
        "important_caution": "Bridge nodes may be controversial or cross-cutting; this does not automatically mean high vote support."
    },
    {
        "metric": "closeness_centrality",
        "plain_meaning": "How close a node is to all other nodes in the network.",
        "project_interpretation": "How quickly a node can reach the rest of the discourse network through relationships.",
        "important_caution": "Sensitive to network density and disconnected components."
    },
    {
        "metric": "eigenvector_centrality",
        "plain_meaning": "Importance based on being connected to important nodes.",
        "project_interpretation": "Prestige-like structural centrality in the discourse network.",
        "important_caution": "Can be unstable or unavailable for some network structures."
    },
])
metric_glossary.to_csv(TABLES / "16_metric_glossary.csv", index=False)
display(metric_glossary)


## 3. Candidate co-mention centrality

This section focuses on the candidate co-mention network. Nodes are candidates. An edge exists when two candidates appear in the same tweet. This measures how candidates are structurally positioned in the conversation, not simply how many votes they received.


In [ ]:
progress.step("3. Candidate co-mention centrality")

def standardize_candidate_metric_table(df, source_label):
    if df.empty:
        return df
    out = df.copy()
    if "node" in out.columns and "candidate" not in out.columns:
        out = out.rename(columns={"node": "candidate"})
    out["network_source"] = source_label
    for col in ["degree", "weighted_degree", "degree_centrality", "pagerank", "betweenness_centrality", "closeness_centrality", "eigenvector_centrality"]:
        if col not in out.columns:
            out[col] = np.nan
    return out

cand_centrality = standardize_candidate_metric_table(candidate_metrics, "candidate_co_mention")

if not cand_centrality.empty:
    rank_cols = ["degree", "weighted_degree", "pagerank", "betweenness_centrality"]
    for col in rank_cols:
        cand_centrality[f"{col}_rank"] = cand_centrality[col].rank(method="min", ascending=False)
    keep_cols = ["candidate", "degree", "weighted_degree", "pagerank", "betweenness_centrality", "degree_rank", "weighted_degree_rank", "pagerank_rank", "betweenness_centrality_rank"]
    cand_rankings = cand_centrality[keep_cols].sort_values(["pagerank", "weighted_degree"], ascending=False)
    cand_rankings.to_csv(TABLES / "16_candidate_comention_centrality_rankings.csv", index=False)
    display(cand_rankings.head(20))
else:
    cand_rankings = pd.DataFrame()
    display(Markdown("**Candidate co-mention metric table is missing. Run notebook 07 first.**"))


### Interpretation guide

- **High degree** means the candidate was mentioned alongside many different candidates.
- **High weighted degree** means those co-mentions were repeated often.
- **High PageRank** means the candidate was connected to other structurally important candidates.
- **High betweenness** means the candidate may connect different candidate-discourse clusters.

This is why centrality should be interpreted as **network position**, not automatically as voter approval.


In [ ]:
progress.step("4. Candidate centrality visual")
if not cand_centrality.empty:
    centrality_melt = cand_centrality.melt(
        id_vars=["candidate"],
        value_vars=["degree", "weighted_degree", "pagerank", "betweenness_centrality"],
        var_name="metric",
        value_name="value"
    ).dropna()
    centrality_melt["rank"] = centrality_melt.groupby("metric")["value"].rank(method="first", ascending=False)
    plot_df = centrality_melt[centrality_melt["rank"] <= 12].copy()
    fig = px.bar(
        plot_df.sort_values(["metric", "value"], ascending=[True, True]),
        x="value",
        y="candidate",
        color="metric",
        orientation="h",
        facet_col="metric",
        facet_col_wrap=2,
        title="Candidate co-mention network centrality: top nodes by metric",
        labels={"value": "Metric value", "candidate": "Candidate"},
        height=900,
    )
    fig.update_yaxes(matches=None, showticklabels=True)
    fig.update_layout(showlegend=False)
    save_plotly(fig, INTERACTIVE / "16_candidate_centrality_deep_dive.html")
    display(fig)


## 4. Bridge and hub hashtags

The hashtag network reveals how election topics and campaign frames cluster. A hashtag can be important because it is frequently used, because it connects to many other hashtags, or because it bridges otherwise separate issue communities.


In [ ]:
progress.step("5. Hashtag hubs and bridges")
if not hashtag_metrics.empty:
    hm = hashtag_metrics.copy()
    if "node" not in hm.columns and "hashtag" in hm.columns:
        hm = hm.rename(columns={"hashtag": "node"})
    for col in ["degree", "weighted_degree", "pagerank", "betweenness_centrality", "community"]:
        if col not in hm.columns:
            hm[col] = np.nan
    hm["degree_rank"] = hm["degree"].rank(method="min", ascending=False)
    hm["weighted_degree_rank"] = hm["weighted_degree"].rank(method="min", ascending=False)
    hm["pagerank_rank"] = hm["pagerank"].rank(method="min", ascending=False)
    hm["betweenness_rank"] = hm["betweenness_centrality"].rank(method="min", ascending=False)
    hashtag_rankings = hm[["node", "degree", "weighted_degree", "pagerank", "betweenness_centrality", "community", "degree_rank", "weighted_degree_rank", "pagerank_rank", "betweenness_rank"]].copy()
    hashtag_rankings.to_csv(TABLES / "16_hashtag_hub_bridge_rankings.csv", index=False)
    display(hashtag_rankings.sort_values("betweenness_centrality", ascending=False).head(20))
else:
    hashtag_rankings = pd.DataFrame()
    display(Markdown("**Hashtag metric table is missing. Run notebook 06 first.**"))


### Interpretation guide

- A **hub hashtag** has many or strong connections to other hashtags.
- A **bridge hashtag** has high betweenness centrality and helps connect otherwise separate conversations.
- In election discourse, bridge hashtags are important because they can join campaign messaging, issue advocacy, media coverage, fandom activity, or controversy into the same public conversation.


In [ ]:
progress.step("6. Hashtag bridge visual")
if not hashtag_metrics.empty:
    bridge_df = hashtag_rankings.sort_values("betweenness_centrality", ascending=False).head(25).sort_values("betweenness_centrality", ascending=True)
    fig = px.bar(
        bridge_df,
        x="betweenness_centrality",
        y="node",
        orientation="h",
        title="Top bridge hashtags by betweenness centrality",
        labels={"betweenness_centrality": "Betweenness centrality", "node": "Hashtag"},
        height=800,
    )
    save_plotly(fig, INTERACTIVE / "16_top_bridge_hashtags_betweenness.html")
    display(fig)


## 5. Candidate–hashtag bipartite centrality

The candidate–hashtag network has two node types: candidate nodes and hashtag nodes. This view asks which candidates are connected to broad hashtag ecosystems and which hashtags connect strongly to candidate discourse.


In [ ]:
progress.step("7. Candidate-hashtag bipartite centrality")
if not candidate_hashtag_metrics.empty:
    bh = candidate_hashtag_metrics.copy()
    if "node_type" not in bh.columns:
        # Older output builds may not have explicit node type. Infer candidate nodes by matching names in the co-mention metric table.
        candidate_name_set = set(cand_centrality["candidate"].astype(str)) if not cand_centrality.empty else set()
        bh["node_type"] = bh["node"].apply(lambda x: "candidate" if str(x) in candidate_name_set else "hashtag")
    for col in ["degree", "weighted_degree", "pagerank", "betweenness_centrality"]:
        if col not in bh.columns:
            bh[col] = np.nan
    bip_candidate_rankings = bh[bh["node_type"].eq("candidate")].copy()
    bip_hashtag_rankings = bh[bh["node_type"].eq("hashtag")].copy()
    bip_candidate_rankings.to_csv(TABLES / "16_candidate_hashtag_candidate_centrality.csv", index=False)
    bip_hashtag_rankings.to_csv(TABLES / "16_candidate_hashtag_hashtag_centrality.csv", index=False)
    display(bip_candidate_rankings.sort_values("pagerank", ascending=False).head(20))
else:
    bip_candidate_rankings = pd.DataFrame()
    bip_hashtag_rankings = pd.DataFrame()
    display(Markdown("**Candidate-hashtag bipartite metrics are missing. Run notebook 08 first.**"))


## 6. Centrality metrics compared with actual votes

This section connects the network-science layer to the election-outcome layer. The goal is not to say that centrality causes votes. The goal is to identify which online network metrics align with vote performance and which metrics reveal online-specific attention patterns.


In [ ]:
progress.step("8. Centrality metrics vs votes")
from scipy.stats import spearmanr, pearsonr

def candidate_key(s):
    return normalize_text(s).replace(" ", "")

centrality_vote = pd.DataFrame()
correlation_rows = []

if not cand_centrality.empty:
    left = cand_centrality.copy()
    left["candidate_key"] = left["candidate"].map(candidate_key)
    if not comparison.empty:
        right = comparison.copy()
    else:
        right = national.copy()
    if not right.empty:
        if "node" in right.columns and "candidate" not in right.columns:
            right = right.rename(columns={"node": "candidate"})
        right["candidate_key"] = right["candidate"].map(candidate_key)
        centrality_vote = left.merge(right, on="candidate_key", how="left", suffixes=("", "_outcome"))
        if "candidate_outcome" in centrality_vote.columns:
            centrality_vote["candidate"] = centrality_vote["candidate"].fillna(centrality_vote["candidate_outcome"])
        centrality_vote.to_csv(TABLES / "16_candidate_centrality_vs_votes.csv", index=False)
        vote_cols = [c for c in ["total_votes", "vote_rank"] if c in centrality_vote.columns]
        metric_cols = ["degree", "weighted_degree", "degree_centrality", "pagerank", "betweenness_centrality", "closeness_centrality", "eigenvector_centrality"]
        for metric in metric_cols:
            for vote_col in vote_cols:
                sub = centrality_vote[[metric, vote_col]].dropna()
                if len(sub) >= 5:
                    sp = spearmanr(sub[metric], sub[vote_col])
                    pe = pearsonr(sub[metric], sub[vote_col])
                    correlation_rows.append({
                        "network": "candidate_co_mention",
                        "network_metric": metric,
                        "outcome_metric": vote_col,
                        "spearman_r": sp.statistic,
                        "spearman_p": sp.pvalue,
                        "pearson_r": pe.statistic,
                        "pearson_p": pe.pvalue,
                        "n_candidates": len(sub),
                        "interpretation_note": "Positive correlation with total_votes means the centrality metric increases with votes. Negative correlation with vote_rank means the metric improves as ranking number gets smaller."
                    })

correlations_deep = pd.DataFrame(correlation_rows)
correlations_deep.to_csv(TABLES / "16_network_metric_vote_correlations.csv", index=False)
if not correlations_deep.empty:
    display(correlations_deep.sort_values(["outcome_metric", "spearman_r"], ascending=[True, False]))
else:
    display(Markdown("**Vote comparison table is missing or insufficient. Run notebooks 01 and 10 first.**"))


### Interpretation guide

A centrality metric can align with votes without being a direct predictor. For example, high weighted degree may mean that a candidate was repeatedly discussed with other candidates. This may reflect campaign strength, controversy, alliance framing, issue salience, or media visibility. The correct interpretation is **alignment between online structure and election outcome**, not causality.


In [ ]:
progress.step("9. Centrality-vote correlation visual")
if not correlations_deep.empty:
    plot_corr = correlations_deep[correlations_deep["outcome_metric"].eq("total_votes")].copy()
    plot_corr = plot_corr.sort_values("spearman_r", ascending=True)
    fig = px.bar(
        plot_corr,
        x="spearman_r",
        y="network_metric",
        orientation="h",
        title="Candidate co-mention centrality metrics vs actual Senate votes",
        labels={"spearman_r": "Spearman correlation with total votes", "network_metric": "Network metric"},
        height=520,
    )
    fig.add_vline(x=0, line_dash="dash")
    save_plotly(fig, INTERACTIVE / "16_centrality_vote_correlation.html")
    display(fig)


## 7. Network-science interpretation summary

The core finding from the metric layer is that online centrality is multidimensional. Some candidates are visible because they are repeatedly mentioned. Others are structurally central because they connect to important candidates or bridge communities. Hashtags can become hubs by frequency, or bridges by connecting separate issue spaces.

This is why a network-science approach is stronger than a simple count of mentions: it shows how attention is organized, not only how much attention exists.


In [ ]:
progress.step("10. Export narrative summary")
summary_lines = []
summary_lines.append("# Network Metrics Deep Dive Summary")
summary_lines.append("")
summary_lines.append("This notebook consolidates the network-science metrics used in the Philippine Election 2025 Twitter/X discourse project.")
summary_lines.append("")
summary_lines.append("## Main metrics interpreted")
summary_lines.append("- Degree: breadth of connection")
summary_lines.append("- Weighted degree: intensity of repeated connection")
summary_lines.append("- PageRank: structural prominence through important neighbors")
summary_lines.append("- Betweenness centrality: bridge or broker position")
summary_lines.append("- Closeness/eigenvector centrality: additional structural position indicators")
summary_lines.append("")
if not cand_rankings.empty:
    top_pr = cand_rankings.sort_values("pagerank", ascending=False).head(5)["candidate"].astype(str).tolist()
    summary_lines.append("## Top candidate PageRank nodes")
    summary_lines.append(", ".join(top_pr))
    summary_lines.append("")
if not hashtag_rankings.empty:
    top_bridge = hashtag_rankings.sort_values("betweenness_centrality", ascending=False).head(5)["node"].astype(str).tolist()
    summary_lines.append("## Top bridge hashtags")
    summary_lines.append(", ".join(top_bridge))
    summary_lines.append("")
summary_lines.append("## Interpretation")
summary_lines.append("Network centrality should be interpreted as online discourse position, not automatically as positive support or voter intent. The value of the metrics is that they reveal how candidates, hashtags, and issue narratives connect across the Twitter/X election conversation.")

summary_path = REPORT_ASSETS / "16_network_metrics_deep_dive_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")
print(f"Saved narrative summary: {summary_path.relative_to(ROOT)}")


## 8. Output inventory

The following files are generated by this notebook and can be used in the report, dashboard, presentation, or GitHub documentation.


In [ ]:
progress.step("11. Output inventory")
expected_outputs = [
    TABLES / "16_metric_glossary.csv",
    TABLES / "16_candidate_comention_centrality_rankings.csv",
    TABLES / "16_hashtag_hub_bridge_rankings.csv",
    TABLES / "16_candidate_hashtag_candidate_centrality.csv",
    TABLES / "16_candidate_hashtag_hashtag_centrality.csv",
    TABLES / "16_candidate_centrality_vs_votes.csv",
    TABLES / "16_network_metric_vote_correlations.csv",
    INTERACTIVE / "16_candidate_centrality_deep_dive.html",
    INTERACTIVE / "16_top_bridge_hashtags_betweenness.html",
    INTERACTIVE / "16_centrality_vote_correlation.html",
    REPORT_ASSETS / "16_network_metrics_deep_dive_summary.md",
]
output_inventory = pd.DataFrame([
    {
        "output": str(p.relative_to(ROOT)),
        "exists": p.exists(),
        "size_kb": round(p.stat().st_size / 1024, 2) if p.exists() else 0,
    }
    for p in expected_outputs
])
display(output_inventory)


## Run complete

This notebook should make the network-science computation visible and defensible in the reproducible notebook pipeline.


In [ ]:
progress.done("Notebook run completed")